# 实验二：PyACL 加载 OM 模型端侧推理

PyACL 是 Python 调用 ACL Runtime 的接口。

本章使用 **YOLOv5s / COCO 80 类 / 640×640 静态输入 / `models/yolov5s_310b4.om`**。输入图片不需要完整 COCO 数据集，只需要少量 COCO 风格样例图用于验证推理链路和采集性能数据。

默认样例图片路径由 `src/configs/yolo_edge.yaml` 的 `data.sample_image` 指定：`src/data/images/bus.jpg`。

## PyACL 推理流程

<table style="margin-left: 0; margin-right: auto; text-align: left;">
  <thead>
    <tr>
      <th style="text-align: left;">阶段</th>
      <th style="text-align: left;">关键动作</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;">初始化</td>
      <td style="text-align: left;"><code>acl.init</code>、<code>acl.rt.set_device</code></td>
    </tr>
    <tr>
      <td style="text-align: left;">加载模型</td>
      <td style="text-align: left;"><code>acl.mdl.load_from_file</code></td>
    </tr>
    <tr>
      <td style="text-align: left;">输入准备</td>
      <td style="text-align: left;">图片 resize/letterbox、NCHW、float32/float16</td>
    </tr>
    <tr>
      <td style="text-align: left;">内存管理</td>
      <td style="text-align: left;">Host/Device buffer 创建、数据拷贝</td>
    </tr>
    <tr>
      <td style="text-align: left;">模型执行</td>
      <td style="text-align: left;"><code>acl.mdl.execute</code></td>
    </tr>
    <tr>
      <td style="text-align: left;">输出读取</td>
      <td style="text-align: left;">Device 到 Host 拷贝，reshape 为 YOLO 输出</td>
    </tr>
    <tr>
      <td style="text-align: left;">资源释放</td>
      <td style="text-align: left;">unload model、reset device、finalize</td>
    </tr>
  </tbody>
</table>

In [ ]:
# ====== 1. 运行预处理函数，确认输入 shape ======
from src.scripts.pyacl_yolo_infer import load_config, preprocess

cfg = load_config('src/configs/yolo_edge.yaml')
image_path = cfg.get('data', {}).get('sample_image', 'src/data/images/bus.jpg')
input_tensor = preprocess(image_path, cfg)
print('image path:', image_path)
print('input shape:', input_tensor.shape)
print('dtype:', input_tensor.dtype)
print('min/max:', float(input_tensor.min()), float(input_tensor.max()))

In [ ]:
# ====== 2. 使用 PyACL Session 模板执行一次推理 ======
# 如果当前环境没有 PyACL 或 OM 文件，会自动进入 dry-run，便于学习流程。
from src.scripts.pyacl_yolo_infer import PyAclYoloSession

with PyAclYoloSession(cfg) as session:
    pred = session.infer(input_tensor)

print('model output shape:', pred.shape)
print('output dtype:', pred.dtype)

## 内存管理注意事项

- 输入输出 buffer 的大小必须与 OM 模型描述一致。
- Host 到 Device、Device 到 Host 的拷贝方向不能写反。
- 频繁分配释放内存会引入额外延迟，端侧应用建议复用 buffer。
- Profiling 时要把模型执行耗时和数据拷贝耗时分开看。

In [ ]:
# ====== 3. 命令行运行方式 ======
print('python src/scripts/pyacl_yolo_infer.py --config src/configs/yolo_edge.yaml --image src/data/images/bus.jpg --repeat 10')

---
## 本章小结

你已经走通了 PyACL 端侧推理应用的代码结构。下一章将展开 YOLO 后处理，并用 CPU NMS 建立优化前的延迟基线。

## 课后练习

请根据本节实验内容完成以下练习。题型包含单选题、多选题、判断题、填空题、简答题和代码设计题。

1. (单选题) PyACL 推理流程中，最先需要完成的运行时步骤通常是？
   - A. acl.init 并设置 device
   - B. 读取输出 buffer
   - C. 执行 CPU NMS
   - D. 打开 MindStudio

2. (单选题) `acl.mdl.load_from_file` 的主要作用是？
   - A. 加载 OM 模型
   - B. 转换 ONNX 模型
   - C. 生成 COCO 标签
   - D. 安装自定义算子

3. (单选题) PyACL 推理中 Host 到 Device 数据拷贝通常发生在哪一步？
   - A. 输入张量准备好后，模型执行前
   - B. 模型执行后读取输出时
   - C. ATC 转换前
   - D. Git push 后

4. (单选题) 如果脚本输出 `using dry-run output`，最准确的含义是？
   - A. 已经真实执行 OM 推理
   - B. 没有真正执行 OM 推理，而是用随机输出帮助流程演示
   - C. NPU 性能已经最优
   - D. 自定义算子已经安装

5. (多选题) PyACL 推理脚本中通常需要管理哪些资源？
   - A. device
   - B. model_id/model_desc
   - C. input/output dataset
   - D. device/host memory buffer

6. (多选题) 判断 PyACL 真实推理跑通，可以看哪些证据？
   - A. 打印 loaded OM model
   - B. used dry-run 为 False
   - C. 输出 shape 与配置一致
   - D. postprocess 能得到检测结果或有效输出

7. (多选题) PyACL 内存管理不当可能导致哪些问题？
   - A. 内存泄漏
   - B. 输出读取失败
   - C. 重复初始化错误
   - D. 模型执行异常

8. (判断题) PyACL 加载 OM 后，仍然需要对输入图片做 resize、归一化和 NCHW 排布等预处理。

9. (判断题) 只要 CPU 后处理能跑通，就能证明 OM 模型一定在 NPU 上执行过。

10. (填空题) PyACL 中执行模型推理的常用接口是 `acl.mdl.____`。

11. (填空题) 本实验 PyACL 推理期望的 YOLO 输出 shape 是 `____`。

12. (简答题) 为什么 PyACL 推理脚本需要显式释放资源？

13. (简答题) PyACL 推理结果为什么还不能直接等同于最终检测框？

14. (简答题) 如果 PyACL 提示 OM 输出 float32 数量少于配置期望，应该检查什么？

15. (代码设计题) 写一段最小伪代码，表示 PyACL 推理的资源顺序。

> 参考答案见 answer/04.03_pyacl_om_inference_answer.ipynb。
